# Анализ транзакций клиентов банка

**Цель исследования:** понять поведение клиентов и транзакций, выявить ценные сегменты и подготовить выводы для продуктовой команды банка (сегментация, приоритеты по возрасту и географии, динамика активности).

**Данные:** транзакции клиентов банка — идентификатор клиента, дата рождения, пол, город, баланс счёта, дата и время транзакции, сумма транзакции. Период покрытия определяется по данным.

**Для кого кейс:** портфолио продуктового и бизнес-аналитика — демонстрация полного цикла: загрузка, очистка, EDA, визуализация и бизнес-выводы с рекомендациями.

## 1. Импорт библиотек и загрузка данных

In [1]:
# --- Импорт библиотек ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Настройка стиля (единая палитра для банковского кейса) ---
sns.set_theme(style="whitegrid", palette="Greens")
pd.set_option("display.max_columns", 50)

# --- Загрузка данных ---
# Подставьте свой путь к файлу. В Colab: загрузите CSV через «Файлы» и укажите имя файла ниже.
FILE_PATH = "bank_clients_transactions.csv"
df = pd.read_csv(FILE_PATH)

# Опционально: установить seaborn, если нет (раскомментируйте при необходимости)
# !pip install -q pandas numpy matplotlib seaborn

df.head(10)

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
df.info()

## 2. Очистка и подготовка данных

In [ ]:
# Приведение имён столбцов к camelCase (удобно для разного написания в исходном CSV)
def to_camel(col):
    parts = str(col).strip().lower().replace(" ", "_").split("_")
    return parts[0] + "".join(p.capitalize() for p in parts[1:])

df.columns = [to_camel(c) for c in df.columns]
df.columns.tolist()

In [ ]:
# Маппинг имён колонок под ваш датасет (при необходимости измените под свои названия)
# Ключ — как называется колонка после to_camel, значение — стандартное имя для кода ниже
COLUMNS = {
    "clientId": ["customerid", "clientid", "client_id"],
    "birthDate": ["customerdob", "birthdate", "birth_date", "dob"],
    "gender": ["custgender", "gender", "sex"],
    "city": ["custlocation", "city", "location", "город"],
    "balance": ["custaccountbalance", "balance", "accountbalance", "баланс"],
    "transactionDate": ["transactiondate", "date", "txdate"],
    "transactionTime": ["transactiontime", "time", "txtime"],
    "transactionAmount": ["transactionamount(Inr)", "transactionamount(inr)", "transactionamount", "amount", "сумма"],
}
# Переименование: находим первую подходящую колонку для каждого стандартного имени
rename_map = {}
for standard_name, variants in COLUMNS.items():
    for v in variants:
        if v in df.columns:
            rename_map[v] = standard_name
            break
df = df.rename(columns=rename_map)
df.columns.tolist()

In [ ]:
# Приведение типов: даты и время
for date_col in ["birthDate", "transactionDate"]:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Числовые поля
for num_col in ["transactionAmount", "balance"]:
    if num_col in df.columns:
        df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

# Время транзакции: если хранится как число (HHMMSS), оставляем для разбора в разделе 9
if "transactionTime" in df.columns and df["transactionTime"].dtype in ["int64", "float64"]:
    pass  # используем как есть для перевода в часы/минуты позже
df.dtypes

In [ ]:
# Обработка дублей и пропусков
n_before = len(df)
df = df.drop_duplicates()
n_duplicates = n_before - len(df)

missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print("Доля пропусков по столбцам (только с пропусками):")
print(missing.round(4).to_string())
print(f"\nУдалено дубликатов: {n_duplicates}")

In [ ]:
# Удаление строк с критическими пропусками: дата транзакции и сумма обязательны
critical_cols = ["transactionAmount"]
if "transactionDate" in df.columns:
    critical_cols.append("transactionDate")
df = df.dropna(subset=critical_cols)
print(f"Строк после удаления по критическим пропускам: {len(df)}")

In [ ]:
# Стандартные имена колонок для всех разделов (зависят от маппинга выше)
client_id_col = "clientId" if "clientId" in df.columns else list(df.columns)[1]
amount_col = "transactionAmount" if "transactionAmount" in df.columns else None
balance_col = "balance" if "balance" in df.columns else None
date_col = "transactionDate" if "transactionDate" in df.columns else None
city_col = "city" if "city" in df.columns else None
gender_col = "gender" if "gender" in df.columns else None
birth_col = "birthDate" if "birthDate" in df.columns else None
time_col = "transactionTime" if "transactionTime" in df.columns else None

**Вывод по очистке:** имена столбцов приведены к единому виду и к стандартным названиям для дальнейшего кода. Даты и числовые поля приведены к нужным типам; дубликаты удалены. Пропуски по столбцам оценены; строки с пропусками в критических полях (дата и сумма транзакции) удалены. Остальные пропуски (например, по возрасту или городу) при необходимости можно учитывать в разбивках.

## 3. Клиенты и транзакции

In [ ]:
client_id_col = globals().get("client_id_col", "clientId" if "clientId" in df.columns else list(df.columns)[1])
n_clients = df[client_id_col].nunique()
n_tx = len(df)
tx_per_client = n_tx / n_clients if n_clients else 0

print(f"Уникальных клиентов:     {n_clients:,}")
print(f"Количество транзакций:   {n_tx:,}")
print(f"Транзакций на клиента:  {tx_per_client:.1f}")

**Вывод:** по числу транзакций на клиента можно судить об уровне активности: среднее значение показывает, насколько часто клиенты совершают операции. Высокое значение говорит об активном использовании продуктов банка, низкое — о сегменте с потенциалом для вовлечения.

## 4. Баланс клиентов и суммы транзакций

In [ ]:
# Баланс: один баланс на клиента (берём последнее значение по транзакциям или первое)
# Fallback на случай запуска не с начала ноутбука
client_id_col = globals().get("client_id_col", df.columns[1] if len(df.columns) > 1 else None)
balance_col = globals().get("balance_col") or ("balance" if "balance" in df.columns else None)
if balance_col:
    client_balance = df.groupby(client_id_col)[balance_col].last().dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(client_balance, bins=50, color="forestgreen", edgecolor="white", alpha=0.85)
    axes[0].set_title("Распределение баланса клиентов")
    axes[0].set_xlabel("Баланс")
    axes[0].set_ylabel("Количество клиентов")
    sns.boxplot(y=client_balance, ax=axes[1], color="mediumseagreen", showfliers=False)
    axes[1].set_title("Баланс клиентов (ящик с усами)")
    axes[1].set_ylabel("Баланс")
    p1, p99 = client_balance.quantile([0.01, 0.99]).values
    axes[1].set_ylim(p1, p99)
    axes[1].tick_params(axis="y", labelsize=9)
    plt.tight_layout()
    plt.show()

In [2]:
# Суммы транзакций (fallback — если ячейку запускают без блока очистки)
amount_col = globals().get("amount_col") or ("transactionAmount" if "transactionAmount" in df.columns else None)
if amount_col:
    amounts = df[amount_col].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(amounts, bins=60, color="forestgreen", edgecolor="white", alpha=0.85)
    axes[0].set_title("Распределение суммы транзакций")
    axes[0].set_xlabel("Сумма транзакции")
    axes[0].set_ylabel("Количество транзакций")
    sns.boxplot(y=amounts, ax=axes[1], color="mediumseagreen", showfliers=False)
    axes[1].set_title("Сумма транзакций (ящик с усами)")
    axes[1].set_ylabel("Сумма транзакций")
    p1, p99 = amounts.quantile([0.01, 0.99]).values
    axes[1].set_ylim(p1, p99)
    axes[1].tick_params(axis="y", labelsize=9)
    plt.tight_layout()
    plt.show()

NameError: name 'amount_col' is not defined

**Вывод:** основная масса балансов и сумм транзакций сосредоточена в нижнем и среднем диапазоне; выбросы (очень высокие балансы или крупные платежи) видны на boxplot. Для бизнеса это означает, что большинство операций — типичные по сумме, а редкие крупные транзакции требуют отдельного внимания (например, премиум-сегмент или антифрод).

## 5. Возраст клиентов

In [ ]:
# Возраст по дате рождения (год транзакции — последний в данных или текущий)
date_col = globals().get("date_col") or ("transactionDate" if "transactionDate" in df.columns else None)
birth_col = globals().get("birth_col") or ("birthDate" if "birthDate" in df.columns else None)
ref_date = df[date_col].max() if date_col else pd.Timestamp.now()
ref_year = ref_date.year if hasattr(ref_date, "year") else pd.Timestamp(ref_date).year

if birth_col:
    df["age"] = (ref_date - df[birth_col]).dt.days / 365.25
    df["age"] = df["age"].clip(lower=0, upper=120)
    df_with_age = df.dropna(subset=["age"])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df_with_age["age"], bins=25, color="forestgreen", edgecolor="white", alpha=0.85)
    ax.set_title("Распределение возраста клиентов")
    ax.set_xlabel("Возраст (лет)")
    ax.set_ylabel("Количество записей")
    plt.tight_layout()
    plt.show()

In [ ]:
# Возрастные группы
if birth_col and "age" in df.columns:
    bins = [0, 24, 34, 44, 59, 150]
    labels = ["18–24", "25–34", "35–44", "45–59", "60+"]
    df["ageGroup"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)
    age_group_counts = df["ageGroup"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 4))
    age_group_counts.plot(kind="bar", ax=ax, color="mediumseagreen", edgecolor="white")
    ax.set_title("Распределение по возрастным группам")
    ax.set_xlabel("Возрастная группа")
    ax.set_ylabel("Количество записей")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()

**Вывод:** по гистограмме и барчарту видно, какие возрастные группы доминируют (часто это 25–44 года). Перекос в сторону молодых или старших клиентов подсказывает, куда ориентировать продукты и коммуникации.

## 6. Пол и география клиентов

In [ ]:
# Распределение по полу (уникальные клиенты)
client_id_col = globals().get("client_id_col", "clientId" if "clientId" in df.columns else list(df.columns)[1])
gender_col = globals().get("gender_col") or ("gender" if "gender" in df.columns else None)
if gender_col:
    clients_gender = df.groupby(client_id_col)[gender_col].first().value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    clients_gender.plot(kind="bar", ax=ax, color=["darkgreen", "mediumseagreen"], edgecolor="white")
    ax.set_title("Распределение клиентов по полу")
    ax.set_xlabel("Пол")
    ax.set_ylabel("Количество клиентов")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Топ-20 городов по числу уникальных клиентов
city_col = globals().get("city_col") or ("city" if "city" in df.columns else None)
if city_col:
    clients_by_city = df.groupby(client_id_col)[city_col].first().value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, 5))
    clients_by_city.sort_values().plot(kind="barh", ax=ax, color="mediumseagreen", edgecolor="white")
    ax.set_title("Топ-20 городов по числу клиентов")
    ax.set_xlabel("Количество клиентов")
    ax.set_ylabel("Город")
    plt.tight_layout()
    plt.show()

**Вывод:** соотношение полов показывает структуру клиентской базы; топ городов по числу клиентов выявляет основные регионы присутствия. Наличие одного-двух доминирующих городов типично для банков с выраженной региональной концентрацией.

## 7. Динамика транзакций и оборота

In [ ]:
# Агрегация по месяцам
date_col = globals().get("date_col") or ("transactionDate" if "transactionDate" in df.columns else None)
amount_col = globals().get("amount_col") or ("transactionAmount" if "transactionAmount" in df.columns else None)
if date_col:
    ts = df.set_index(pd.to_datetime(df[date_col]))
    monthly_count = ts.resample("M").size()
    monthly_sum = ts.resample("M")[amount_col].sum() if amount_col else None

    fig, ax = plt.subplots(figsize=(10, 4))
    monthly_count.plot(ax=ax, color="darkgreen", linewidth=2)
    ax.set_title("Динамика количества транзакций по месяцам")
    ax.set_xlabel("Месяц")
    ax.set_ylabel("Число транзакций")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
if date_col and amount_col:
    fig, ax = plt.subplots(figsize=(10, 4))
    monthly_sum.plot(ax=ax, color="darkgreen", linewidth=2)
    ax.set_title("Динамика суммы транзакций по месяцам")
    ax.set_xlabel("Месяц")
    ax.set_ylabel("Сумма транзакций")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

**Вывод:** по графикам виден общий тренд (рост или снижение активности и оборота), возможная сезонность (пики и провалы по месяцам). Сравнение динамики количества и суммы транзакций показывает, меняется ли средний чек во времени.

## 8. Возрастные и географические паттерны

In [ ]:
# Сумма транзакций по возрастным группам
amount_col = globals().get("amount_col") or ("transactionAmount" if "transactionAmount" in df.columns else None)
if "ageGroup" in df.columns and amount_col:
    by_age = df.groupby("ageGroup", observed=True)[amount_col].sum().sort_index()
    fig, ax = plt.subplots(figsize=(8, 4))
    by_age.plot(kind="bar", ax=ax, color="mediumseagreen", edgecolor="white")
    ax.set_title("Сумма транзакций по возрастным группам")
    ax.set_xlabel("Возрастная группа")
    ax.set_ylabel("Сумма транзакций")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Топ-10 городов по сумме транзакций
city_col = globals().get("city_col") or ("city" if "city" in df.columns else None)
amount_col = globals().get("amount_col") or ("transactionAmount" if "transactionAmount" in df.columns else None)
if city_col and amount_col:
    by_city = df.groupby(city_col)[amount_col].sum().nlargest(10)
    fig, ax = plt.subplots(figsize=(9, 4))
    by_city.sort_values().plot(kind="barh", ax=ax, color="mediumseagreen", edgecolor="white")
    ax.set_title("Топ-10 городов по сумме транзакций")
    ax.set_xlabel("Сумма транзакций")
    ax.set_ylabel("Город")
    plt.tight_layout()
    plt.show()

**Вывод:** возрастная группа с максимальной суммой транзакций — ключевой сегмент по выручке; топ городов по обороту показывает, куда эффективнее направлять маркетинг и развитие продуктов. На основе этого можно предложить таргетирование по возрасту и географии.

## 9. Время суток транзакций

In [ ]:
# Время транзакции: если в данных число формата HHMMSS (например 143207 = 14:32:07)
time_col = globals().get("time_col") or ("transactionTime" if "transactionTime" in df.columns else None)
date_col = globals().get("date_col") or ("transactionDate" if "transactionDate" in df.columns else None)
if time_col:
    t = df[time_col].dropna().astype(int)
    hours = (t // 10000) % 24
    minutes = (t // 100) % 60
    df["hour_of_day"] = hours + minutes / 60
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df["hour_of_day"].dropna(), bins=24, color="forestgreen", edgecolor="white", alpha=0.85)
    ax.set_title("Распределение времени транзакций по суткам")
    ax.set_xlabel("Час от начала суток")
    ax.set_ylabel("Количество транзакций")
    plt.tight_layout()
    plt.show()
else:
    # Если есть только дата — можно построить по часам из datetime
    if date_col and pd.api.types.is_datetime64_any_dtype(df[date_col]):
        df["hour_of_day"] = pd.to_datetime(df[date_col]).dt.hour
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(df["hour_of_day"].dropna(), bins=24, color="forestgreen", edgecolor="white", alpha=0.85)
        ax.set_title("Распределение времени транзакций по суткам (по часу)")
        ax.set_xlabel("Час")
        ax.set_ylabel("Количество транзакций")
        plt.tight_layout()
        plt.show()

**Вывод:** пики активности по времени суток показывают, когда клиенты чаще всего совершают операции; эти окна наиболее подходят для промо-акций и планирования нагрузки на системы.

## 10. Итоговые выводы

По результатам анализа транзакций и клиентов банка можно сформулировать следующие выводы для продуктовой и маркетинговой команд.

**Портрет клиента и активность:** основной объём клиентской базы составляют клиенты с умеренным числом транзакций; распределение по полу и возрасту показывает, какие демографические сегменты преобладают (часто это возраст 25–44 года и один из полов). Уровень активности (транзакций на клиента) характеризует вовлечённость и потенциал для кросс-продаж.

**Балансы и суммы операций:** большая часть балансов и сумм транзакций лежит в среднем диапазоне; наличие выбросов указывает на сегмент с высоким балансом или крупными платежами, который целесообразно выделять отдельно (премиум, антифрод, персональные предложения).

**Возрастные и географические паттерны:** возрастная группа с максимальной суммой транзакций даёт основной вклад в оборот и должна быть в фокусе продуктовых и маркетинговых инициатив. Топ городов по числу клиентов и по сумме транзакций определяет ключевые регионы присутствия; при расхождениях между «топ по клиентам» и «топ по обороту» можно корректировать приоритеты по регионам.

**Динамика и время:** тренд по месяцам (рост/падение) и возможная сезонность важны для планирования кампаний и ресурсов. Распределение транзакций по времени суток выявляет пиковые часы для коммуникаций и технической нагрузки.

**Рекомендации:** (1) таргетировать продукты и акции на возрастную группу с наибольшим оборотом и на топовые города по сумме транзакций; (2) углубить сегментацию по балансу и среднему чеку (низкий/средний/премиум) для персонализации предложений; (3) использовать динамику по месяцам и по времени суток для планирования промо и развития каналов (мобильное приложение, push, поддержка).